# Blackjack CV — Análisis de Partidas

Este notebook analiza el historial de manos guardado en `data/games_log.csv`.
Si el CSV no existe o está vacío, ejecuta primero:
```bash
python scripts/gen_sample_data.py --hands 300
```

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from src.analysis.stats import (
    load_log, print_summary,
    plot_bankroll, plot_outcomes, plot_delta_by_upcard,
    plot_action_distribution, plot_adherence_by_session,
    plot_player_total_distribution,
)

LOG_PATH = '../data/games_log.csv'

%matplotlib inline
plt.rcParams['figure.dpi'] = 110

## 1. Carga de datos

In [ ]:
df = load_log(LOG_PATH)
print(f'{len(df)} manos cargadas')
df.head()

## 2. Resumen de la sesión

In [ ]:
print_summary(df)

## 3. Evolución del bankroll

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
plot_bankroll(df, ax=ax)
plt.tight_layout()
plt.show()

## 4. Distribución de resultados y EV por carta del crupier

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
plot_outcomes(df, ax=ax1)
plot_delta_by_upcard(df, ax=ax2)
plt.tight_layout()
plt.show()

**Lectura:** el gráfico derecho muestra el delta medio (ganancia/pérdida) según la carta visible del crupier.
Las cartas 2-6 son favorables para el jugador; 9, 10 y As son las más peligrosas.

## 5. Acciones y totales iniciales del jugador

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
plot_action_distribution(df, ax=ax1)
plot_player_total_distribution(df, ax=ax2)
plt.tight_layout()
plt.show()

## 6. Adherencia a la estrategia por sesión

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
plot_adherence_by_session(df, ax=ax)
plt.tight_layout()
plt.show()
print('(100% = seguiste basic strategy en todas las manos)')

## 7. Análisis libre

In [ ]:
# EV por tipo de mano (soft / hard)
from src.game.card import Card
from src.game.hand import Hand

def hand_type(cards_str: str) -> str:
    try:
        h = Hand([Card(r) for r in str(cards_str).split()])
        if h.is_blackjack():   return 'blackjack'
        if h.is_soft():        return 'soft'
        return 'hard'
    except Exception:
        return 'unknown'

df['hand_type'] = df['player_cards'].apply(hand_type)

ev_by_type = df.groupby('hand_type')['delta'].agg(['mean', 'count'])
ev_by_type.columns = ['EV medio (€)', 'Manos']
ev_by_type.sort_values('EV medio (€)', ascending=False)

In [ ]:
# Racha máxima de victorias y derrotas consecutivas
outcomes = df['outcome'].map(lambda o: 1 if o in ('ganas','blackjack') else (-1 if o=='pierdes' else 0))

def max_streak(series, value):
    max_s = cur = 0
    for v in series:
        if v == value:
            cur += 1
            max_s = max(max_s, cur)
        else:
            cur = 0
    return max_s

print(f'Racha máxima de victorias  : {max_streak(outcomes, 1)}')
print(f'Racha máxima de derrotas   : {max_streak(outcomes, -1)}')